# Backtest (low-level API)

Tutorial for [NautilusTrader](https://nautilustrader.io/docs/latest/) a high-performance algorithmic trading platform and event-driven backtester.

[View source on GitHub](https://github.com/nautechsystems/nautilus_trader/blob/develop/docs/getting_started/backtest_low_level.ipynb).

## Overview

This tutorial walks through how to use a `BacktestEngine` to backtest a simple EMA cross strategy
with a TWAP execution algorithm on a simulated Binance Spot exchange using historical trade tick data.

The following points will be covered:
- Load raw data (external to Nautilus) using data loaders and wranglers.
- Add this data to a `BacktestEngine`.
- Add venues, strategies, and execution algorithms to a `BacktestEngine`.
- Run backtests with a `BacktestEngine`.
- Perform post-run analysis and repeated runs.


## Prerequisites
- Python 3.12+ installed.
- [JupyterLab](https://jupyter.org/) or similar installed (`uv pip install jupyterlab`).
- [NautilusTrader](https://pypi.org/project/nautilus_trader/) latest release installed (`uv pip install nautilus_trader`).

## Imports

We'll start with all of our imports for the remainder of this tutorial.

In [2]:
from decimal import Decimal

from nautilus_trader.backtest.config import BacktestEngineConfig
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.examples.algorithms.twap import TWAPExecAlgorithm
from nautilus_trader.examples.strategies.ema_cross_twap import EMACrossTWAP
from nautilus_trader.examples.strategies.ema_cross_twap import EMACrossTWAPConfig
from nautilus_trader.model import BarType
from nautilus_trader.model import Money
from nautilus_trader.model import TraderId
from nautilus_trader.model import Venue
from nautilus_trader.model.currencies import ETH
from nautilus_trader.model.currencies import USDT
from nautilus_trader.model.enums import AccountType
from nautilus_trader.model.enums import OmsType
from nautilus_trader.persistence.wranglers import TradeTickDataWrangler
from nautilus_trader.test_kit.providers import TestDataProvider
from nautilus_trader.test_kit.providers import TestInstrumentProvider

## Loading data

For this tutorial we use stub test data from the NautilusTrader repository (the automated test suite also uses this data to verify platform correctness).

First, instantiate a data provider to read raw CSV trade tick data into memory as a `pd.DataFrame`.
Next, initialize the instrument that matches the data (in this case the `ETHUSDT` spot cryptocurrency pair for Binance) and reuse it for the remainder of the backtest run.

Then wrangle the data into a list of Nautilus `TradeTick` objects so you can add them to the `BacktestEngine`.


In [ ]:
#Load stub test data
provider = TestDataProvider()
trades_df = provider.read_csv_ticks("binance/ethusdt-trades.csv")
# Initialize the instrument which matches the data
ETHUSDT_BINANCE = TestInstrumentProvider.ethusdt_binance()

# Process into Nautilus objects
wrangler = TradeTickDataWrangler(instrument=ETHUSDT_BINANCE)
ticks = wrangler.process(trades_df)

Couldn't find test data directory, test data will be pulled from GitHub
                                   trade_id   price  quantity  buyer_maker
timestamp                                                                 
2020-08-14 10:00:00.223000+00:00  148568980  423.76   2.67900         True
2020-08-14 10:00:00.976000+00:00  148568981  423.74   2.31976         True
2020-08-14 10:00:00.976000+00:00  148568982  423.73   2.16924         True
2020-08-14 10:00:01.185000+00:00  148568983  423.68   0.19096         True
2020-08-14 10:00:01.913000+00:00  148568984  423.70   0.82490        False
...                                     ...     ...       ...          ...
2020-08-14 14:59:58.450000+00:00  148638711  426.88   0.49000         True
2020-08-14 14:59:58.544000+00:00  148638712  426.92   0.50198        False
2020-08-14 14:59:58.544000+00:00  148638713  426.93   1.97786        False
2020-08-14 14:59:58.544000+00:00  148638714  426.94   0.86133        False
2020-08-14 14:59:58.693000+0

See the [Loading External Data](https://nautilustrader.io/docs/latest/concepts/data#loading-data) guide for a more detailed explanation of the typical data processing components and pipeline.

## Initialize a backtest engine

Create a backtest engine. You can call `BacktestEngine()` to instantiate an engine with the default configuration.

We also initialize a `BacktestEngineConfig` (with only a custom `trader_id` specified) to illustrate the general configuration pattern.

See the [Configuration](https://nautilustrader.io/docs/api_reference/config) API reference for details of all configuration options available.


In [11]:
# Configure backtest engine
config = BacktestEngineConfig(trader_id=TraderId("BACKTESTER-001"))

# Build the backtest engine
engine = BacktestEngine(config=config)

## Add venues

Create a venue to trade on that matches the market data you add to the engine.

In this case we set up a simulated Binance Spot exchange.


In [12]:
# Add a trading venue (multiple venues possible)
BINANCE = Venue("BINANCE")
engine.add_venue(
    venue=BINANCE,
    oms_type=OmsType.NETTING,
    account_type=AccountType.CASH,  # Spot CASH account (not for perpetuals or futures)
    base_currency=None,  # Multi-currency account
    starting_balances=[Money(1_000_000.0, USDT), Money(10.0, ETH)],
)

## Add data

Add data to the backtest engine. Start by adding the `Instrument` object we initialized earlier to match the data.

Then add the trades we wrangled earlier.


In [13]:
# Add instrument(s)
engine.add_instrument(ETHUSDT_BINANCE)

# Add data
engine.add_data(ticks)

:::note
Machine resources and your imagination limit the amount and variety of data types you can use (custom types are possible).
You can also backtest across multiple venues, again constrained only by machine resources.
:::


## Add strategies

Add the trading strategies you plan to run as part of the system.

:::note
You can backtest multiple strategies and instruments; machine resources remain the only limit.
:::

First, initialize a strategy configuration, then use it to initialize a strategy that you can add to the engine:


In [14]:
# Configure your strategy
strategy_config = EMACrossTWAPConfig(
    instrument_id=ETHUSDT_BINANCE.id,
    bar_type=BarType.from_str("ETHUSDT.BINANCE-250-TICK-LAST-INTERNAL"),
    trade_size=Decimal("0.10"),
    fast_ema_period=10,
    slow_ema_period=20,
    twap_horizon_secs=10.0,
    twap_interval_secs=2.5,
)

# Instantiate and add your strategy
strategy = EMACrossTWAP(config=strategy_config)
engine.add_strategy(strategy=strategy)

You may notice that this strategy config includes parameters related to a TWAP execution algorithm.
We can flexibly use different parameters per order submit, but we still need to initialize and add the actual `ExecAlgorithm` component that executes the algorithm.

## Add execution algorithms

NautilusTrader enables you to build complex systems of custom components. Here we highlight one built-in component: a TWAP execution algorithm. Configure it and add it to the engine using the same general pattern as for strategies.

:::note
You can backtest multiple execution algorithms; machine resources remain the only limit.
:::


In [15]:
# Instantiate and add your execution algorithm
exec_algorithm = TWAPExecAlgorithm()  # Using defaults
engine.add_exec_algorithm(exec_algorithm)

## Run backtest

After configuring the data, venues, and trading system, run a backtest.
Call the `.run(...)` method to process all available data by default.

See the [BacktestEngineConfig](https://nautilustrader.io/docs/latest/api_reference/config) API reference for a complete description of all available methods and options.


In [16]:
# Run the engine (from start to end of data)
engine.run()

## Post-run and analysis

After the backtest completes, the engine automatically logs a post-run tearsheet with default statistics (or custom statistics that you load; see the advanced [Portfolio statistics](../concepts/portfolio.md#portfolio-statistics) guide).

The engine also keeps many data and execution objects in memory, which you can use to generate additional reports for performance analysis.


In [17]:
engine.trader.generate_account_report(BINANCE)

,total,locked,free,currency,account_id,account_type,base_currency,margins,reported,info
2020-08-14 10:00:00.223000+00:00,1000000.00000000,0.00000000,1000000.00000000,USDT,BINANCE-001,CASH,None,[],True,{}
2020-08-14 10:00:00.223000+00:00,10.00000000,0.00000000,10.00000000,ETH,BINANCE-001,CASH,None,[],True,{}
2020-08-14 10:22:34.574000+00:00,999989.38468857,0.00000000,999989.38468857,USDT,BINANCE-001,CASH,None,[],False,{}
2020-08-14 10:22:34.574000+00:00,10.02500000,0.00000000,10.02500000,ETH,BINANCE-001,CASH,None,[],False,{}
2020-08-14 10:22:37.074000+00:00,999978.76937714,0.00000000,999978.76937714,USDT,BINANCE-001,CASH,None,[],False,{}
...,...,...,...,...,...,...,...,...,...,...
2020-08-14 14:24:40.817000+00:00,10.07500000,0.00000000,10.07500000,ETH,BINANCE-001,CASH,None,[],False,{}
2020-08-14 14:24:43.317000+00:00,999956.20097365,0.00000000,999956.20097365,USDT,BINANCE-001,CASH,None,[],False,{}
2020-08-14 14:24:43.317000+00:00,10.10000000,0.00000000,10.10000000,ETH,BINANCE-001,CASH,None,[],False,{}
2020-08-14 14:59:58.693000+00:00,999998.88570475,0.00000000,999998.88570475,USDT,BINANCE-001,CASH,None,[],False,{}


In [18]:
engine.trader.generate_order_fills_report()

,trader_id,strategy_id,instrument_id,venue_order_id,position_id,account_id,last_trade_id,type,side,quantity,...,order_list_id,linked_order_ids,parent_order_id,exec_algorithm_id,exec_algorithm_params,exec_spawn_id,tags,init_id,ts_init,ts_last
client_order_id,,,,,,,,,,,,,,,,,,,,,
O-20200814-102234-001-000-1,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-004,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-004,MARKET,BUY,0.02500,...,None,None,None,TWAP,"{'horizon_secs': 10.0, 'interval_secs': 2.5}",O-20200814-102234-001-000-1,None,88dbb2bc-8ffa-4764-95ff-ff9d82afcf89,2020-08-14 10:22:34.574000+00:00,2020-08-14 10:22:42.074000+00:00
O-20200814-102234-001-000-1-E1,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-001,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-001,MARKET,BUY,0.02500,...,None,None,None,TWAP,None,O-20200814-102234-001-000-1,None,ecc38787-0353-4c44-ab2e-75bf0d6b4614,2020-08-14 10:22:34.574000+00:00,2020-08-14 10:22:34.574000+00:00
O-20200814-102234-001-000-1-E2,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-002,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-002,MARKET,BUY,0.02500,...,None,None,None,TWAP,None,O-20200814-102234-001-000-1,None,5bbc8136-aca5-447f-8955-40396a82215a,2020-08-14 10:22:37.074000+00:00,2020-08-14 10:22:37.074000+00:00
O-20200814-102234-001-000-1-E3,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-003,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-003,MARKET,BUY,0.02500,...,None,None,None,TWAP,None,O-20200814-102234-001-000-1,None,2d23e428-e2c4-442a-bedf-9e42d3387e11,2020-08-14 10:22:39.574000+00:00,2020-08-14 10:22:39.574000+00:00
O-20200814-103237-001-000-2,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-005,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-005,MARKET,SELL,0.10000,...,None,None,None,None,None,None,None,3603fe79-a32d-4955-a3b7-1d8c68f78b56,2020-08-14 10:32:37.428000+00:00,2020-08-14 10:32:37.428000+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
O-20200814-142435-001-000-29,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-074,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-075,MARKET,BUY,0.02500,...,None,None,None,TWAP,"{'horizon_secs': 10.0, 'interval_secs': 2.5}",O-20200814-142435-001-000-29,None,d1f8f93d-9545-4059-b63d-319a38130cec,2020-08-14 14:24:35.817000+00:00,2020-08-14 14:24:43.317000+00:00
O-20200814-142435-001-000-29-E1,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-071,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-072,MARKET,BUY,0.02500,...,None,None,None,TWAP,None,O-20200814-142435-001-000-29,None,a46032d9-35c1-4b10-814b-858f706ed65b,2020-08-14 14:24:35.817000+00:00,2020-08-14 14:24:35.817000+00:00
O-20200814-142435-001-000-29-E2,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-1-072,ETHUSDT.BINANCE-EMACrossTWAP-000,BINANCE-001,BINANCE-1-073,MARKET,BUY,0.02500,...,None,None,None,TWAP,None,O-20200814-142435-001-000-29,None,9a7600a9-2a30-4bbc-8406-959e087dd989,2020-08-14 14:24:38.317000+00:00,2020-08-14 14:24:38.317000+00:00


In [19]:
engine.trader.generate_positions_report()

,trader_id,strategy_id,instrument_id,account_id,opening_order_id,closing_order_id,entry,side,quantity,peak_qty,...,ts_opened,ts_last,ts_closed,duration_ns,avg_px_open,avg_px_close,commissions,realized_return,realized_pnl,is_snapshot
position_id,,,,,,,,,,,,,,,,,,,,,
ETHUSDT.BINANCE-EMACrossTWAP-000-dfacbca9-59e9-42c4-b1fa-ae1c5cebbc59,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-102234-001-000-1-E1,O-20200814-103237-001-000-2,BUY,FLAT,0.00000,0.10000,...,2020-08-14 10:22:34.574000+00:00,1597401157428000000,2020-08-14 10:32:37.428000+00:00,602854000000,424.5625,423.490000,[0.00848054 USDT],-0.00253,-0.11573054 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-050196bc-23d2-45f3-b4d7-c5aaf13917ec,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-103237-001-000-3-E1,O-20200814-104611-001-000-4,SELL,FLAT,0.00000,0.10000,...,2020-08-14 10:32:37.428000+00:00,1597401971428000000,2020-08-14 10:46:11.428000+00:00,814000000000,423.4875,424.700000,[0.00848189 USDT],-0.00286,-0.12973189 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-cba1a4e4-7649-4eed-876d-bda3c4e3726a,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-104611-001-000-5-E1,O-20200814-110002-001-000-6,BUY,FLAT,0.00000,0.10000,...,2020-08-14 10:46:11.428000+00:00,1597402802097000000,2020-08-14 11:00:02.097000+00:00,830669000000,424.6050,424.143982,[0.00848750 USDT],-0.00109,-0.05458930 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-eaa1dac9-d881-40fa-9f36-85a0a968f18e,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-110002-001-000-7-E1,O-20200814-111402-001-000-8,SELL,FLAT,0.00000,0.10000,...,2020-08-14 11:00:02.097000+00:00,1597403642429000000,2020-08-14 11:14:02.429000+00:00,840332000000,424.0050,425.570000,[0.00849575 USDT],-0.00369,-0.16499575 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-5a2eb1eb-5a90-4c0c-9ba6-eb08de382c6b,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-111402-001-000-9-E1,O-20200814-113300-001-000-10,BUY,FLAT,0.00000,0.10000,...,2020-08-14 11:14:02.429000+00:00,1597404780084000000,2020-08-14 11:33:00.084000+00:00,1137655000000,425.6375,425.340000,[0.00850979 USDT],-0.00070,-0.03825979 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-8d8f8395-adf2-4afd-bc44-bff72489458e,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-113300-001-000-11-E1,O-20200814-113705-001-000-12,SELL,FLAT,0.00000,0.10000,...,2020-08-14 11:33:00.084000+00:00,1597405025204000000,2020-08-14 11:37:05.204000+00:00,245120000000,425.4350,426.250000,[0.00851685 USDT],-0.00192,-0.09001685 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-5f8278c2-3b2d-4804-b91f-f80c9fbb3558,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-113705-001-000-13-E1,O-20200814-120215-001-000-14,BUY,FLAT,0.00000,0.07500,...,2020-08-14 11:37:05.204000+00:00,1597406535235000000,2020-08-14 12:02:15.235000+00:00,1510031000000,426.3100,426.430000,[0.00639556 USDT],0.00028,0.00260444 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-2ca2845c-d7a6-480e-b37d-fbe0acdc59ac,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-120215-001-000-15-E1,O-20200814-123931-001-000-16,SELL,FLAT,0.00000,0.10000,...,2020-08-14 12:02:15.235000+00:00,1597408771933000000,2020-08-14 12:39:31.933000+00:00,2236698000000,426.3600,427.330000,[0.00853691 USDT],-0.00228,-0.10553691 USDT,True
ETHUSDT.BINANCE-EMACrossTWAP-000-e9f6f9e2-799b-42da-96bf-4439be32593d,BACKTESTER-001,EMACrossTWAP-000,ETHUSDT.BINANCE,BINANCE-001,O-20200814-123931-001-000-17-E1,O-20200814-125346-001-000-18,BUY,FLAT,0.00000,0.10000,...,2020-08-14 12:39:31.933000+00:00,1597409626057000000,2020-08-14 12:53:46.057000+00:00,854124000000,427.3225,426.260000,[0.00853583 USDT],-0.00249,-0.11478583 USDT,True


## Repeated runs

You can reset the engine for repeated runs with different strategy and component configurations.

Instruments and data persist across resets by default, so you don't need to reload them.

In [20]:
# For repeated backtest runs, reset the engine
engine.reset()

# Instruments and data persist, just add new components and run again

Remove and add individual components (actors, strategies, execution algorithms) as required.

See the [Trader](../api_reference/trading.md) API reference for a description of all methods available to achieve this.


In [ ]:
# Once done, good practice to dispose of the object if the script continues
engine.dispose()